In [0]:
from pyspark.sql import functions as F

CATALOG = "workspace"
SCHEMA = "colloky"
RAW_PATH = "/Volumes/workspace/colloky/raw/colloky_snapshots.csv.csv"

spark.sql(f"USE {CATALOG}.{SCHEMA}")

DataFrame[]

In [0]:
df_raw = (
    spark.read.format("csv")
    .option("header", True)          # la primera fila son los nombres de columnas
    .option("multiLine", True)       # las descripciones pueden tener saltos de línea
    .option("quote", '"')
    .option("escape", '"')
    .option("inferSchema", False)    # Bronze guarda todo como texto, sin interpretar
    .load(RAW_PATH)
    .select("*", F.col("_metadata.file_path").alias("_source_file"))
    .withColumn("_ingested_at", F.current_timestamp())
)

print(f"Filas: {df_raw.count():,}")
print(f"Columnas: {len(df_raw.columns)}")
display(df_raw.limit(10))

Filas: 53,469
Columnas: 42


id,date_scraped,brand,product_url,category,page_number,page_position,category_rank,sku,sku_number,image_url,product_name,product_name_raw,product_description,description_main,care_tips,when_to_use,pairs_with,product_type,gender,age_segment,age_min_months,size_range,is_licensed,license_brand,license_character,material,material_source,material_composition,color,color_type,color_count,original_price,final_price,discount,offers_total,offers_in_stock,offers_out_of_stock,stockout_rate,created_at,_source_file,_ingested_at
1295,2026-10-06 18:42:00,Colloky,https://www.colloky.cl/abrigo-coleccion-azul-marino-nino-abpa0157i26,Clothing,22,12,642,56355,56355,https://colgramcl.vtexassets.com/arquivos/ids/915801-500-750?v=639056652437230000&width=500&height=750&aspect=true,Abrigo niño azul marino 18/24M a 6/7A,Abrigo niño azul marino 18/24M a 6/7A,"El abrigo azul marino de niño es una prenda esencial para mantener a tu pequeño abrigado y cómodo durante los días más fríos. Este abrigo está confeccionado con una mezcla de materiales que favorecen el desarrollo infantil, permitiendo que los niños se muevan libremente mientras juegan y exploran su entorno. La estructura de gabardina proporciona una resistencia adecuada, ideal para el uso diario, mientras que el cierre de botones asegura un ajuste perfecto y fácil de poner y quitar. La manga larga ofrece una cobertura adicional, protegiendo los brazos de las inclemencias del tiempo. Este abrigo es perfecto para la etapa infantil plus, ya que se adapta a las necesidades de los niños en crecimiento, brindando la calidez necesaria sin restringir sus movimientos. Además, su color azul marino es versátil y combina fácilmente con otras prendas, permitiendo crear diferentes looks para cada ocasión. La prenda está diseñada para ser fácil de cuidar, lo que es un gran alivio para los padres ocupados. Al elegir este abrigo, no solo estás optando por una prenda funcional, sino también por una que acompaña a tu hijo en su desarrollo, asegurando que se sienta seguro y protegido mientras explora el mundo que lo rodea. ¡nos encanta acompañarte en cada etapa!","El abrigo azul marino de niño es una prenda esencial para mantener a tu pequeño abrigado y cómodo durante los días más fríos. Este abrigo está confeccionado con una mezcla de materiales que favorecen el desarrollo infantil, permitiendo que los niños se muevan libremente mientras juegan y exploran su entorno. La estructura de gabardina proporciona una resistencia adecuada, ideal para el uso diario, mientras que el cierre de botones asegura un ajuste perfecto y fácil de poner y quitar. La manga larga ofrece una cobertura adicional, protegiendo los brazos de las inclemencias del tiempo. Este abrigo es perfecto para la etapa infantil plus, ya que se adapta a las necesidades de los niños en crecimiento, brindando la calidez necesaria sin restringir sus movimientos. Además, su color azul marino es versátil y combina fácilmente con otras prendas, permitiendo crear diferentes looks para cada ocasión. La prenda está diseñada para ser fácil de cuidar, lo que es un gran alivio para los padres ocupados. Al elegir este abrigo, no solo estás optando por una prenda funcional, sino también por una que acompaña a tu hijo en su desarrollo, asegurando que se sienta seguro y protegido mientras explora el mundo que lo rodea. ¡nos encanta acompañarte en cada etapa!",NULL,NULL,NULL,Abrigo,Boy,Toddler (1-2Y),18,18/24m a 6/7a,False,Own Brand,None,"Viscose, Polyester, Elastane",specification,2% Elastano 95% Poliester 3% Viscosa,Azul Marino,Single,1,39990,19995,0.5000,5,3,2,0.4000,2026-10-06 18:49:06.629661,dbfs:/Volumes/workspace/colloky/raw/colloky_snapshots.csv.csv,2026-10-07T22:56:38.040Z
1296,2026-10-06 18:42:00,Colloky,https://www.colloky.cl/abrigo-coleccion-gris-nina-abpa0206i26,Clothing,22,13,643,51958,51958,https://colgramcl.vtexassets.com/arquivos/ids/918547-500-750?v=639063633524570000&width=500&height=750&aspect=true,Abrigo cuello niña gris 18/24M a 6/7A,Abrigo cuello ni

In [0]:
(
    df_raw.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("colloky_bronze_snapshots")
)

print("✅ Tabla colloky_bronze_snapshots creada")

✅ Tabla colloky_bronze_snapshots creada


In [0]:
%sql
SELECT
    date_scraped,
    COUNT(*) AS products
FROM colloky_bronze_snapshots
GROUP BY date_scraped
ORDER BY date_scraped

date_scraped,products
2026-10-06 18:42:00,3379
2026-10-06 19:35:00,3439
2026-10-06 20:30:00,3383
2026-10-06 21:25:00,3432
2026-10-06 22:27:00,3438
2026-10-06 23:51:00,3429
2026-10-07 09:24:00,3204
2026-10-07 15:06:00,3257
2026-10-07 16:01:00,3266
2026-10-07 17:05:00,3314
